# 01 — Coleta e integração

**Dengue no Nordeste brasileiro — 2000–2025 · IMD1151, 2026.2**

Este notebook extrai ou reutiliza a base tratada do PostgreSQL e a integra à população e ao cadastro municipal do mesmo snapshot. A etapa termina com painéis agregados e verificações de integridade; não reestima testes nem altera as regras epidemiológicas.

Fontes originais: SINAN/OpenDataSUS e IBGE/SIDRA/Localidades.

O notebook anterior foi preservado em `historico/notebooks/01_coleta_integracao_pipeline_anterior.ipynb`. O notebook `02_limpeza_eda.ipynb` ainda pertence ao pipeline anterior e não é executado por esta etapa.

## 1. Ambiente e configuração

Execute com o ambiente `.venv` do repositório. As dependências de extração estão em `database/requirements-extracao.txt`; pandas e Jupyter constam de `requirements.txt`.

Por padrão, este notebook aproveita a extração concluída mais recente e verifica seus hashes. Para consultar novamente o banco, use `REEXTRAIR = True` e configure `DENGUE_ENV` no ambiente, apontando para um `.env` privado, ou use `.env` na raiz. O banco precisa estar em execução. A conexão de extração é somente de leitura, com isolamento REPEATABLE READ. Senhas não são exibidas nem salvas no notebook.

In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidatos if (p / 'scripts/extrair_postgresql.py').is_file())
sys.path.insert(0, str(ROOT / 'scripts'))
from integrar_postgresql import localizar_manifesto, conferir_arquivos, integrar

REEXTRAIR = False
ARQUIVO_ENV = Path(os.environ.get('DENGUE_ENV', str(ROOT / '.env')))
print('Python:', sys.version.split()[0])
display(pd.DataFrame([{'biblioteca': nome, 'versao': importlib.metadata.version(nome)}
                      for nome in ['pandas', 'psycopg', 'python-dotenv', 'nbclient']]))

Python: 3.14.6


,biblioteca,versao
0,pandas,2.3.3
1,psycopg,3.3.9
2,python-dotenv,1.2.4
3,nbclient,0.11.0


## 2. Coleta a partir do banco

A extração mantém todas as notificações tratadas do período, incluindo descartados e possíveis duplicatas sinalizados. `caso_provavel` é o numerador principal do relatório; total de notificações e casos confirmados são medidas distintas. A geografia corresponde ao município de residência.

Os CSVs individuais ficam em `data/raw/postgresql/`, fora do Git. O manifesto registra o snapshot, as consultas SQL, a data e as contagens. Se não houver cache concluído, a célula abaixo tenta uma extração usando a configuração privada disponível.

In [2]:
try:
    MANIFESTO = localizar_manifesto(ROOT)
except FileNotFoundError:
    MANIFESTO = None

if REEXTRAIR or MANIFESTO is None:
    if not ARQUIVO_ENV.is_file():
        raise FileNotFoundError('Configure DENGUE_ENV ou .env privado para extrair do PostgreSQL.')
    subprocess.run([sys.executable, '-B', str(ROOT / 'scripts/extrair_postgresql.py'),
                    '--env', str(ARQUIVO_ENV)], cwd=ROOT, check=True)
    MANIFESTO = localizar_manifesto(ROOT)
    modo = 'nova extração somente de leitura'
else:
    modo = 'cache de uma extração PostgreSQL concluída; integridade será conferida'

print('Modo desta execução:', modo)
print('Manifesto:', MANIFESTO.relative_to(ROOT).as_posix())

Modo desta execução: cache de uma extração PostgreSQL concluída; integridade será conferida
Manifesto: data/metadata/postgresql/relatorio_2000_2025/20261113_221719/manifesto.json


## 3. Proveniência e integridade dos insumos

São verificados os cinco arquivos de entrada, o resumo anual e o dicionário. As contagens no manifesto foram conciliadas com o banco na extração. A integração abaixo fará uma nova contagem dos registros e das sete métricas anuais ao ler os CSVs em blocos.

In [3]:
manifesto, arquivos = conferir_arquivos(ROOT, MANIFESTO)
display(pd.DataFrame([{
    'banco': manifesto['banco'], 'schema': manifesto['schema'],
    'periodo': '2000–2025', 'extraido_em': manifesto['concluido_em'],
    'somente_leitura_na_extracao': manifesto['somente_leitura'],
    'snapshot': manifesto['snapshot']
}]))
display(arquivos)
dicionario = json.loads((ROOT / manifesto['dicionario']['arquivo']).read_text(encoding='utf-8'))
display(pd.DataFrame(dicionario))

,banco,schema,periodo,extraido_em,somente_leitura_na_extracao,snapshot
0,dengue-eda,public,2000–2025,2026-11-13T22:18:51.708987-03:00,True,902:902:


,tabela,registros,sha256_conferido
0,notificacao_dengue_ne,5474491,True
1,populacao_municipio_ano,46627,True
2,populacao_ne,43056,True
3,municipio_ne,1794,True
4,estado_ne,9,True


,tabela,coluna,tipo_sql,tipo_interno,permite_nulo,comprimento_maximo
0,estado,uf_cod,smallint,int2,NO,NaN
1,estado,sigla,character,bpchar,NO,2.0
2,estado,nome,text,text,NO,NaN
3,estado,regiao_sigla,character,bpchar,NO,2.0
4,estado,regiao,text,text,NO,NaN
5,municipio,cod_mun7,character,bpchar,NO,7.0
6,municipio,cod_mun6,character,bpchar,NO,6.0
7,municipio,nome,text,text,NO,NaN
8,municipio,uf_cod,smallint,int2,NO,NaN
9,municipio,microrregiao,text,text,YES,NaN


## 4. Integração por município e ano

1. As notificações são lidas em blocos de 200 mil linhas e agregadas por `cod_mun7` e `ano`.
2. Municípios e UFs são ligados por `uf_cod`, com validação de cardinalidade; notificações e população usam o código IBGE de sete dígitos e o mesmo ano.
3. O painel preserva municípios-ano com população e nenhum caso, preenchendo apenas as contagens com zero. A junção externa também conserva eventuais notificações sem denominador, que são auditadas.
4. A população já tratada do banco é mantida, incluindo a fonte `interpolada` em 2007 e 2023. Nenhum município recebe 26 anos de cobertura por pressuposto.
5. A incidência é `casos_provaveis / populacao × 100.000`, com denominador positivo. Possíveis duplicatas continuam incluídas e sinalizadas; esta etapa não as remove.

Os sete totais anuais são comparados ao resumo obtido diretamente do PostgreSQL. Não são exibidos registros individuais.

In [4]:
resultado = integrar(ROOT, MANIFESTO)
validacao = resultado['validacao']
print('Integração:', validacao['status'])
print('Comprovante:', resultado['caminho_validacao'])
display(pd.DataFrame([{'metrica': k, 'total': v} for k, v in validacao['totais'].items()]))

Blocos processados: 5 (1,000,000 registros)


Blocos processados: 10 (2,000,000 registros)


Blocos processados: 15 (3,000,000 registros)


Blocos processados: 20 (4,000,000 registros)


Blocos processados: 25 (5,000,000 registros)


Integração: concluido
Comprovante: data/metadata/postgresql/relatorio_2000_2025/20261113_221719/integracoes/20261113_222826.json


,metrica,total
0,notificacoes,5474491
1,casos_provaveis,4178450
2,casos_confirmados,2884120
3,possiveis_duplicatas,101804
4,sem_idade,10816
5,sem_sexo,6682
6,sem_mes_sintomas,12928


## 5. Painéis integrados e cobertura

As tabelas abaixo são agregadas. O painel regional soma os numeradores e denominadores estaduais antes de calcular a taxa; não faz média simples das taxas municipais. A cobertura populacional pode variar conforme o ano e a criação dos municípios.

In [5]:
regiao = resultado['painel_regiao_ano']
ufs = resultado['painel_uf_ano']
cobertura = resultado['cobertura']
display(regiao[['ano', 'notificacoes', 'casos_provaveis', 'casos_confirmados',
                'populacao', 'incidencia_provaveis_100mil']])
display(cobertura.groupby('uf', as_index=False).agg(
    primeiro_ano=('ano', 'min'), ultimo_ano=('ano', 'max'),
    menor_cobertura=('municipios_com_populacao', 'min'),
    maior_cobertura=('municipios_com_populacao', 'max'),
    municipio_anos_interpolados=('municipios_interpolados', 'sum')))
display(ufs.loc[ufs['ano'].eq(2025), ['uf','ano','notificacoes','casos_provaveis',
                                     'populacao','incidencia_provaveis_100mil']])

,ano,notificacoes,casos_provaveis,casos_confirmados,populacao,incidencia_provaveis_100mil
0,2000,83283,70187,50929,47782486,146.888548
1,2001,180931,149734,95902,48331186,309.808247
2,2002,326105,267759,182581,48845112,548.179724
3,2003,207668,151541,109463,49352225,307.060117
4,2004,41850,22368,17025,50427274,44.356949
5,2005,124662,76039,61463,51019091,149.040288
6,2006,107714,69757,59183,51609027,135.164339
7,2007,175517,125608,91581,52345293,239.960449
8,2008,299716,209255,148698,53088496,394.162607
9,2009,177789,126763,83696,53591197,236.536982


,uf,primeiro_ano,ultimo_ano,menor_cobertura,maior_cobertura,municipio_anos_interpolados
0,AL,2000,2025,101,102,204
1,BA,2000,2025,415,417,834
2,CE,2000,2025,184,184,368
3,MA,2000,2025,217,217,434
4,PB,2000,2025,223,223,446
5,PE,2000,2025,185,185,370
6,PI,2000,2025,221,224,447
7,RN,2000,2025,166,167,334
8,SE,2000,2025,75,75,150


,uf,ano,notificacoes,casos_provaveis,populacao,incidencia_provaveis_100mil
25,MA,2025,5672,5672,7018211,80.818317
51,PI,2025,9289,9289,3384547,274.453272
77,CE,2025,5454,5454,9268836,58.842340
103,RN,2025,9480,9480,3455236,274.366208
129,PB,2025,7769,7769,4164468,186.554441
155,PE,2025,21050,21050,9562007,220.142069
181,AL,2025,8096,8096,3220848,251.362374
207,SE,2025,712,712,2299425,30.964263
233,BA,2025,32434,32434,14870907,218.103711


## 6. Reconciliação e arquivos de saída

A execução deve falhar se houver divergências de hashes, chaves duplicadas, registros fora do recorte, perda de contagens, duplicação de população ou casos prováveis sem denominador. As taxas mantêm a precisão numérica dos painéis; as tabelas exibidas podem arredondar apenas a apresentação.

In [6]:
assert validacao['totais'] == manifesto['totais']
assert validacao['casos_provaveis_sem_populacao'] == 0
assert len(regiao) == 26 and len(ufs) == 234
print('Notificações sem população:', validacao['notificacoes_sem_populacao'])
print('Casos prováveis sem população:', validacao['casos_provaveis_sem_populacao'])
display(pd.DataFrame([{'verificacao': v, 'resultado': 'aprovada'}
                      for v in validacao['verificacoes']]))
display(pd.DataFrame([{'tabela': k, **v} for k,v in validacao['arquivos'].items()]))

Notificações sem população: 0
Casos prováveis sem população: 0


,verificacao,resultado
0,SHA-256 das entradas,aprovada
1,chaves unicas,aprovada
2,juncoes validadas,aprovada
3,recorte temporal e geografico,aprovada
4,sete metricas por ano conciliadas com PostgreSQL,aprovada
5,populacao conservada na juncao,aprovada
6,26 anos e 234 UF-anos,aprovada
7,casos provaveis sem denominador igual a zero,aprovada


,tabela,arquivo,linhas,sha256
0,painel_municipio_ano,data/processed/postgresql/relatorio_2000_2025/...,46627,23101d31e040bff0df1a68eec6b542f74e648153e5ec53...
1,painel_uf_ano,data/processed/postgresql/relatorio_2000_2025/...,234,f5a99ef3fd55cbff9a8f0bda440a02621e8bfd38bb4ee6...
2,painel_regiao_ano,data/processed/postgresql/relatorio_2000_2025/...,26,b2d8d741c6ba54c587747016d571f7d3e9900e507820d9...
3,cobertura_populacao_uf_ano,data/processed/postgresql/relatorio_2000_2025/...,234,c2ea5731e12bc95b90afcec4b9bf24ca83d336ed51ee54...


## 7. Resultado desta etapa e limites

A coleta pelo PostgreSQL e a integração resultam em quatro tabelas: município-ano, UF-ano, região-ano e cobertura populacional. Cada execução grava uma pasta própria e um comprovante com hashes, preservando as anteriores. Os arquivos agregados e os metadados podem ser versionados; os registros individuais permanecem locais.

Esta etapa conserva o tratamento já aplicado no banco. Ela não valida externamente a definição de caso nem elimina subnotificação, mudanças da ficha do SINAN ou diferenças na investigação entre UFs. A incidência usa o numerador operacional do projeto e 2025 permanece sujeito a revisão. Licenças e decisões pendentes do grupo continuam abertas.

A EDA e os testes das perguntas P1–P4 pertencem às etapas posteriores. Consulte `database/reproducao/.claude/specs/03_decisoes.md`, `docs/extracao_postgresql.md` e o registro de IA em `.claude/specs/99_uso_ia.md`.